In [32]:
import duckdb

con = duckdb.connect()

con.sql("""
    CREATE TABLE matches_clean AS
    SELECT
        m.season.id AS season_id,
        m.matchday AS match_day,
        m.id AS match_id,
        m.utcDate AS date,
        m.homeTeam.name AS home_team,
        m.awayTeam.name AS away_team,
        m.score.winner AS winner
    FROM (
        SELECT unnest(matches) AS m
        FROM read_json_auto('data/raw/matches.json')
    )

""")

con.sql("SELECT * FROM matches_clean").show()

┌───────────┬───────────┬──────────┬─────────────────────┬───────────────────────────┬───────────────────────────┬───────────┐
│ season_id │ match_day │ match_id │        date         │         home_team         │         away_team         │  winner   │
│   int64   │   int64   │  int64   │      timestamp      │          varchar          │          varchar          │  varchar  │
├───────────┼───────────┼──────────┼─────────────────────┼───────────────────────────┼───────────────────────────┼───────────┤
│      2502 │         1 │   560542 │ 2026-08-21 19:00:00 │ Arsenal FC                │ Coventry City FC          │ HOME_TEAM │
│      2502 │         1 │   560543 │ 2026-08-22 11:30:00 │ Hull City AFC             │ Manchester United FC      │ HOME_TEAM │
│      2502 │         1 │   560544 │ 2026-08-22 14:00:00 │ Ipswich Town FC           │ Sunderland AFC            │ HOME_TEAM │
│      2502 │         1 │   560545 │ 2026-08-22 14:00:00 │ Nottingham Forest FC      │ Leeds United FC         

In [33]:
con.sql("""
    CREATE TABLE form_last_five_games AS
    WITH x as (
    Select
        match_id,
        date,
        home_team as team,
        away_team as opponent,
        case when winner = 'HOME_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'HOME_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE

    Union all

    Select
        match_id,
        date,
        away_team as team,
        home_team as opponent,
        case when winner = 'AWAY_TEAM' then 'win'
            when winner = 'DRAW' then 'draw'
            else 'loss'
        end as result,
        case when winner = 'AWAY_TEAM' then 3
            when winner = 'DRAW' then 1
            else 0
        end as points
    From matches_clean
    Where date <= CURRENT_DATE
    ),

    ranked as (
        select *,
            row_number() over (partition by team order by date desc) as rn
        from x
    )

    select team, sum(points) as form
    from ranked
    where rn <= 5
    group by team
""")

con.sql("SELECT * FROM form_last_five_games").show()

┌───────────────────────────┬────────┐
│           team            │  form  │
│          varchar          │ int128 │
├───────────────────────────┼────────┤
│ Crystal Palace FC         │      4 │
│ Coventry City FC          │      3 │
│ Manchester City FC        │     15 │
│ Leeds United FC           │      9 │
│ Tottenham Hotspur FC      │      2 │
│ Aston Villa FC            │      4 │
│ Arsenal FC                │     12 │
│ Ipswich Town FC           │      6 │
│ Newcastle United FC       │      8 │
│ Fulham FC                 │      2 │
│ Sunderland AFC            │      4 │
│ Nottingham Forest FC      │      5 │
│ Brentford FC              │      9 │
│ Chelsea FC                │      7 │
│ Brighton & Hove Albion FC │     10 │
│ Liverpool FC              │      9 │
│ AFC Bournemouth           │      3 │
│ Hull City AFC             │      8 │
│ Everton FC                │      9 │
│ Manchester United FC      │      5 │
└───────────────────────────┴────────┘
  20 rows                

In [34]:
con.sql("SELECT winner, count(*) FROM matches_clean GROUP BY ALL").show()
con.sql("SELECT min(date), max(date), typeof(date), count(*) FROM matches_clean").show()

┌───────────┬──────────────┐
│  winner   │ count_star() │
│  varchar  │    int64     │
├───────────┼──────────────┤
│ AWAY_TEAM │           16 │
│ HOME_TEAM │           18 │
│ DRAW      │           16 │
│ NULL      │          330 │
└───────────┴──────────────┘



BinderException: Binder Error: column "date" must appear in the GROUP BY clause or must be part of an aggregate function.
Either add it to the GROUP BY list, or use "ANY_VALUE(date)" if the exact value of "date" is not important.

In [ ]:
con.sql("SELECT COUNT(*), MIN(date), MAX(date) FROM matches_clean").show()

┌──────────────┬─────────────────────┬─────────────────────┐
│ count_star() │      min(date)      │      max(date)      │
│    int64     │      timestamp      │      timestamp      │
├──────────────┼─────────────────────┼─────────────────────┤
│           38 │ 2026-08-21 19:00:00 │ 2027-05-30 15:00:00 │
└──────────────┴─────────────────────┴─────────────────────┘

